# Mean / Median Imputation

The most common way to fill missing numeric values: replace them with the column's own **mean** or **median**. Simple and keeps the column's overall average roughly unchanged — but as this notebook shows, it comes with a real cost to the column's **variance** and **shape**.

## 1. Setup and Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

In [3]:
df = pd.read_csv('titanic_toy.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'titanic_toy.csv'

In [ ]:
df.head()

In [ ]:
df.info()

## 2. Split Before Imputing

Same principle as scaling/encoding — learn the mean/median from training data only.

In [ ]:
df.isnull().mean()

In [ ]:
X = df.drop(columns=['Survived'])
y = df['Survived']

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=2)

## 3. Computing and Applying the Fill Values

**Mean** vs. **median**: median is the safer default when a column has outliers or skew (e.g. `Fare`, pulled up by a few expensive tickets) — the mean would be dragged toward those outliers, the median wouldn't.

In [ ]:
X_train.shape, X_test.shape

In [ ]:
X_train.isnull().mean()

In [ ]:
mean_age = X_train['Age'].mean()
median_age = X_train['Age'].median()

mean_fare = X_train['Fare'].mean()
median_fare = X_train['Fare'].median()

## 4. The Real Cost — Variance Shrinkage

Every value that gets imputed with the **same fixed number** (the mean/median) collapses toward that number — artificially shrinking the column's spread. The more missing values there are, the bigger this effect.

In [ ]:
X_train['Age_median'] = X_train['Age'].fillna(median_age)
X_train['Age_mean'] = X_train['Age'].fillna(mean_age)

X_train['Fare_median'] = X_train['Fare'].fillna(median_fare)
X_train['Fare_mean'] = X_train['Fare'].fillna(mean_fare)

### Visualizing the Distortion

An obvious spike right at the mean/median value (where none existed before) is the direct visual signature of this variance shrinkage — every imputed row stacks up at exactly one point.

In [ ]:
X_train.sample(5)

In [ ]:
print('Original Age variable variance: ', X_train['Age'].var())
print('Age Variance after median imputation: ', X_train['Age_median'].var())
print('Age Variance after mean imputation: ', X_train['Age_mean'].var())

print('Original Fare variable variance: ', X_train['Fare'].var())
print('Fare Variance after median imputation: ', X_train['Fare_median'].var())
print('Fare Variance after mean imputation: ', X_train['Fare_mean'].var())

### Checking Relationships Between Columns

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)

# original variable distribution
X_train['Age'].plot(kind='kde', ax=ax)

# variable imputed with the median
X_train['Age_median'].plot(kind='kde', ax=ax, color='red')

# variable imputed with the mean
X_train['Age_mean'].plot(kind='kde', ax=ax, color='green')

# add legends
lines, labels = ax.get_legend_handles_labels()
ax.legend(lines, labels, loc='best')

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)

# original variable distribution
X_train['Fare'].plot(kind='kde', ax=ax)

# variable imputed with the median
X_train['Fare_median'].plot(kind='kde', ax=ax, color='red')

# variable imputed with the mean
X_train['Fare_mean'].plot(kind='kde', ax=ax, color='green')

# add legends
lines, labels = ax.get_legend_handles_labels()
ax.legend(lines, labels, loc='best')

### Boxplots — Spotting New "Outliers"

Mean/median imputation can also distort a boxplot's whiskers and quartiles — worth checking that imputation didn't create the *appearance* of new outliers, or hide real ones.

In [ ]:
X_train.cov()

In [ ]:
X_train.corr()

In [ ]:
X_train[['Age', 'Age_median', 'Age_mean']].boxplot()

## 5. Doing the Same Thing With `SimpleImputer`

`strategy='median'` / `strategy='mean'` — the reusable, train/test-safe version via `ColumnTransformer`.

In [ ]:
X_train[['Fare', 'Fare_median', 'Fare_mean']].boxplot()

### Using Sklearn

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=2)

In [ ]:
imputer1 = SimpleImputer(strategy='median')
imputer2 = SimpleImputer(strategy='mean')

`.statistics_` shows the mean/median each imputer actually learned from `X_train`.

In [ ]:
trf = ColumnTransformer([
    ('imputer1',imputer1,['Age']),
    ('imputer2',imputer2,['Fare'])
],remainder='passthrough')

In [ ]:
trf.fit(X_train)

In [ ]:
trf.named_transformers_['imputer1'].statistics_

In [ ]:
trf.named_transformers_['imputer2'].statistics_

## Summary

| Task | Method |
|---|---|
| Fill with mean/median (manual) | `df['col'].fillna(df['col'].mean())` / `.median()` |
| Fill with mean/median (sklearn, reusable) | `SimpleImputer(strategy='mean')` / `'median'` |
| Check variance shrinkage | `.var()` before/after |
| Check distribution distortion | overlaid `.plot(kind='kde')` |
| Check for new/hidden outliers | `.boxplot()` before vs. after |

### Mean vs. median
Use **median** whenever the column is skewed or has outliers (it isn't pulled by them); mean is fine for a roughly symmetric column. Either way, more missing values → more variance shrinkage — worth checking `.isnull().mean()` first to know how much this actually matters here.

In [ ]:
X_train = trf.transform(X_train)
X_test = trf.transform(X_test)